# Replay a NASDAQ ITCH feed with Lobo

The replay API keeps parsing and book reconstruction in Rust while exposing messages as a native Polars `LazyFrame`. This means the lazy plan can use the full Polars API before it is marked for replay with `.adapt()`.

In [ ]:
import os
from datetime import time
from pathlib import Path

import polars as pl

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "Cargo.toml").is_file() and (p / "python/lobo").is_dir()
)

data = Path(
    os.environ.get(
        "LOBO_ITCH_PATH",
        ROOT / "data/NASDAQ/01302020.NASDAQ_ITCH50",
    )
)
data

## Inspect and adapt the source

`source.to_lazy_frame()` represents every ITCH message. `source.adapt()` selects the message types that can update an order book without materializing the feed. Sources are reusable, so either lazy plan can be collected more than once.

In [ ]:
from lobo.replay.adapters import itch

source = itch(data)
all_messages = source.to_lazy_frame()
msgs = source.adapt()

{
    "all_messages": all_messages.collect_schema(),
    "replay_messages": msgs.to_lazy_frame().collect_schema(),
}

## Build a replay run with Polars

`as_lazy_frame()` returns an ordinary typed Polars lazy frame—not a narrowed query wrapper. Filters, expressions, joins, sorting, and the rest of the Polars lazy API remain available. Call `.adapt()` on the final plan to validate its replay columns and turn it back into `ReplayMessages`.

In [ ]:
run = (
    msgs.as_lazy_frame()
    .filter(
        pl.col("timestamp").is_between(
            time(8, 0, 0),
            time(8, 5, 0),
        )
    )
    .sort(["timestamp", "tracking_number"])
    .adapt()
)
run

## Reconstruct books

Entering `ReplayContext` streams the lazy result into the Rust replay engine and constructs one book per symbol. Symbols are looked up case-insensitively. A replay book has the same read-oriented shape used by `PyBook`, including `best_bid`, `best_ask`, and `orders`.

In [ ]:
from lobo.replay import ReplayContext

with ReplayContext(adapter=itch, msgs=run) as books:
    apple_book = books["aapl"]
    replay_summary = {
        "symbols": books.keys(),
        "message_count": books.message_count,
        "skipped_message_count": books.skipped_message_count,
        "aapl_best_bid": apple_book.best_bid,
        "aapl_best_ask": apple_book.best_ask,
    }

replay_summary

In [ ]:
from lobo.levels import levels_lazy

levels_lazy(apple_book.orders.bids).collect()